# 4장 (실습 3): 코드 저장소의 하드코딩된 API Key 찾기

> 책 『AI 보안, 클라우드에서 시작한다』 4장 [실습] 3번을 노트북에서 단계별로 실행하는 버전입니다. 소요 시간은 30분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 코드에서 지운 API Key가 **Git 이력에는 그대로 남는다**는 것을 Gitleaks로 확인합니다.
2. 이미 커밋한 키를 처리하는 순서(**폐기 → 이력 제거 → 재발 방지**)를 이해합니다.
3. 커밋 전에 키를 막는 **pre-commit 훅**을 직접 걸어 보고, 키가 든 커밋이 거부되는 것을 확인합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | 예정 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | 예정 |
| 2 | 실수를 "고친" 척하기 | 예정 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | 예정 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | 예정 |
| 5 | 재발 방지: 커밋 전에 막기 | 예정 |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

## 실습 시나리오

API·RAG 서비스를 만드는 개발자가 급하게 AI 서비스 키를 설정 파일에 넣고 커밋했습니다. 나중에 실수를 알아채고 키를 환경변수로 옮겼습니다. 지금 코드에는 키가 없으니 안전해졌을까요? (4-2의 "API Key 관리")

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Gitleaks 8.19 이상이 필요합니다. 설치되어 있지 않으면 0단계 셀이 `~/bin`에 내려받습니다. 6단계(선택)는 Docker로 TruffleHog을 실행합니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> - 2장 노트북([ch02_lab1_container_isolation.ipynb](../ch02-container-security/ch02_lab1_container_isolation.ipynb))의 실습 전 확인사항과 같습니다. `~/.venv-lab`에 jupyterlab을 설치한 뒤 이 폴더에서 `~/.venv-lab/bin/jupyter lab`을 실행합니다.
> - 노트북은 **반드시 `ch04-ai-usage-types` 폴더에서 열어야** 합니다. 실습 파일을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Git 2.43.0, Gitleaks 8.24.0, Docker 29.1.3에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch04_lab3_gitleaks_api_key_executed.ipynb`입니다.

> ⚠️ **주의**
> **반드시 가짜 키로만 실습하세요.** 실제 키를 저장소에 올리면 몇 분 안에 자동 스캐너에 수집될 수 있습니다. 이 노트북은 홈 디렉터리의 `~/leak-lab`에 실습용 저장소를 따로 만들며, 원격 저장소에는 올리지 않습니다.

## 0. 실습 준비: Gitleaks 확인

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | 예정 |
| 2 | 실수를 "고친" 척하기 | 예정 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | 예정 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | 예정 |
| 5 | 재발 방지: 커밋 전에 막기 | 예정 |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

노트북 전체에서 쓸 도우미 함수(`sh()`, `table()`)를 만들고 도구를 확인합니다. 실습용 저장소는 이 노트북 폴더가 아니라 `~/leak-lab`에 만듭니다. 이 노트북이 있는 저장소 안에 또 다른 Git 저장소를 만들면 헷갈리기 때문입니다.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "git": "git --version",
    "gitleaks": "gitleaks version",
    "docker": "docker --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

REPO = Path.home() / "leak-lab"   # 실습용 저장소 위치
print("\n실습용 저장소:", REPO)

Gitleaks가 ❌로 나오면 아래 셀로 설치합니다. 공식 배포 파일을 내려받고 체크섬을 확인합니다(`OK`가 나와야 합니다). 이미 ✅라면 건너뜁니다.

In [ ]:
%%bash
command -v gitleaks >/dev/null && { echo "이미 설치됨: $(gitleaks version)"; exit 0; }
mkdir -p ~/bin && cd /tmp
V=8.24.0
curl -fsSLO https://github.com/gitleaks/gitleaks/releases/download/v$V/gitleaks_${V}_linux_x64.tar.gz
curl -fsSLO https://github.com/gitleaks/gitleaks/releases/download/v$V/gitleaks_${V}_checksums.txt
grep linux_x64.tar.gz gitleaks_${V}_checksums.txt | sha256sum -c
tar -xzf gitleaks_${V}_linux_x64.tar.gz -C ~/bin gitleaks
gitleaks version

## 1. 실습용 저장소 만들고 가짜 키 커밋하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 실수를 "고친" 척하기 | 예정 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | 예정 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | 예정 |
| 5 | 재발 방지: 커밋 전에 막기 | 예정 |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

실습용 저장소를 만들고, 가짜 키가 들어간 설정 파일을 커밋합니다.

> 📌 (참고) 커밋하려면 작성자 이름과 이메일이 필요합니다. PC 전체 설정(`--global`)을 바꾸지 않도록, 이 저장소에만 가짜 값을 넣습니다. 노트북을 다시 실행할 때를 위해 기존 `~/leak-lab`은 지우고 시작합니다.

In [ ]:
%%bash
rm -rf ~/leak-lab && mkdir ~/leak-lab && cd ~/leak-lab
git init -q -b main
git config user.name "실습생"
git config user.email "student@example.com"

# 가짜 키가 들어간 설정 파일 (실제 키 아님)
cat > config.py << 'EOF'
# AI 서비스 연동 설정
LLM_ENDPOINT = "https://api.example-ai.com/v1/chat"
api_key = "sk-proj-4f9Kd8Lm2Qx7Rt1Vb6Nz3Hy5Wc0Pj8Ga"
EOF

git add config.py
git commit -q -m "AI 연동 추가"
git log --oneline

## 2. 실수를 "고친" 척하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | 예정 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | 예정 |
| 5 | 재발 방지: 커밋 전에 막기 | 예정 |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

개발자가 실수를 알아채고 키를 지웠습니다. 키는 환경변수에서 읽도록 바꿨습니다.

In [ ]:
%%bash
cd ~/leak-lab
cat > config.py << 'EOF'
import os
LLM_ENDPOINT = "https://api.example-ai.com/v1/chat"
api_key = os.environ["LLM_API_KEY"]   # 환경변수에서 읽기
EOF

git commit -q -am "키를 환경변수로 이동"
git log --oneline
echo "── 지금 파일 내용"
cat config.py

지금 파일에는 키가 없습니다. 안전해졌을까요? 다음 셀처럼 Git은 **지난 커밋의 내용을 그대로 보관**합니다. 저장소를 복제한 사람은 누구나 이 명령으로 키를 볼 수 있습니다.

In [ ]:
%%bash
cd ~/leak-lab
git log -p --format='── 커밋 %h %s' -- config.py | grep -E '^(──|[-+]api_key)'

## 3. 현재 파일 스캔과 이력 전체 스캔 비교

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | ✅ 완료 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | 예정 |
| 5 | 재발 방지: 커밋 전에 막기 | 예정 |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 3.1. 현재 파일만 검사 (`gitleaks dir`)

이력은 보지 않고 지금 디렉터리의 파일만 검사합니다.

In [ ]:
%%bash
cd ~/leak-lab
gitleaks dir -v --no-banner --no-color . 2>&1 | tail -3

### 3.2. 저장소 이력 전체 검사 (`gitleaks git`)

모든 커밋을 처음부터 검사합니다. Gitleaks 8.19 이상의 명령이며, 이전 버전에서는 `gitleaks detect -v --source .`를 씁니다. 키가 발견되면 Gitleaks는 종료 코드 1을 돌려주므로 셀 끝에 종료 코드를 출력합니다.

In [ ]:
%%bash
cd ~/leak-lab
gitleaks git -v --no-banner --no-color . 2>&1
echo "종료 코드: $?"

키가 들어 있던 **첫 커밋의 ID, 파일과 줄 번호, 탐지 규칙 이름(`generic-api-key`)** 이 나옵니다. 보고서를 JSON으로 받아 파이썬으로 정리해 봅니다.

> 📌 (참고) Gitleaks는 기본으로 찾은 비밀 값을 그대로 보여 줍니다. 보고서를 다른 사람과 공유할 때는 `--redact` 옵션으로 값을 가리세요. 아래 셀은 `--redact`를 붙였습니다.

In [ ]:
def gitleaks_report(mode, path):
    """gitleaks를 mode(dir/git)로 실행해 찾은 항목 목록을 돌려줍니다."""
    out = Path("/tmp/gitleaks-report.json")
    sh(f"cd {path} && gitleaks {mode} --redact -f json -r {out} . 2>/dev/null")
    return json.loads(out.read_text() or "[]")


rows = []
for mode, label in [("dir", "현재 파일만 (`gitleaks dir`)"), ("git", "저장소 이력 전체 (`gitleaks git`)")]:
    found = gitleaks_report(mode, REPO)
    where = ", ".join(f"{f['File']}:{f['StartLine']} 커밋 {f.get('Commit', '')[:7] or '-'}" for f in found) or "-"
    rows.append((label, f"**{len(found)}건**", where))
table(["스캔 방식", "찾은 키", "위치"], rows)

found = gitleaks_report("git", REPO)
if found:
    print(json.dumps({k: found[0][k] for k in ["RuleID", "Description", "File", "StartLine", "Commit", "Message", "Secret"]},
                     ensure_ascii=False, indent=2))

`[표 실습4-5]` 스캔 범위에 따른 결과 비교

| 스캔 방식 | 키 발견 여부 | 의미 |
|---|---|---|
| 현재 파일만 (`gitleaks dir`) | 발견 안 됨 | 지금 코드에는 키가 없음 |
| 저장소 이력 전체 (`gitleaks git`) | **발견됨** | 첫 번째 커밋에 키가 그대로 남아 있음 |

> 📌 (참고) 탐지 여부는 Gitleaks 버전과 규칙에 따라 다를 수 있습니다. 가짜 키가 탐지되지 않으면 `api_key = "..."` 형태로 다시 쓰거나, 더 긴 무작위 문자열로 바꿔 보세요. 일반 키 규칙(`generic-api-key`)은 "키 이름 + 무작위성이 높은 값"을 찾습니다.

### 3.3. 무작위성(엔트로피)이 하는 일

`generic-api-key` 규칙은 이름이 `api_key`라고 해서 무조건 잡지 않습니다. 값이 **충분히 무작위해야** 키로 봅니다. 같은 이름에 값만 바꿔 직접 확인해 봅니다.

In [ ]:
import math
from collections import Counter


def entropy(s):
    """문자열의 섀넌 엔트로피(글자당 비트)"""
    n = len(s)
    return -sum(c / n * math.log2(c / n) for c in Counter(s).values())


tmp = Path("/tmp/entropy-test")
tmp.mkdir(exist_ok=True)
samples = ["sk-proj-4f9Kd8Lm2Qx7Rt1Vb6Nz3Hy5Wc0Pj8Ga", "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa", "changeme", "your-api-key-here"]
rows = []
for value in samples:
    (tmp / "config.py").write_text(f'api_key = "{value}"\n')
    hits = gitleaks_report("dir", tmp)
    rows.append((f"`{value}`", f"{abs(entropy(value)):.2f}", "탐지" if hits else "-"))
table(["값", "엔트로피(비트/글자)", "Gitleaks"], rows)

무작위한 키는 엔트로피가 높아 탐지되고, `aaaa…`나 `changeme` 같은 값은 탐지되지 않습니다. 오탐을 줄이는 장치이지만, 반대로 **사람이 만든 단순한 비밀번호는 놓칠 수 있다**는 뜻이기도 합니다.

## 4. 이미 커밋된 키는 어떻게 처리하나

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | ✅ 완료 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | ✅ 완료 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 재발 방지: 커밋 전에 막기 | 예정 |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

순서가 중요합니다.

1. **키부터 폐기·재발급합니다.** 이력에서 지우는 것보다 먼저입니다. 원격 저장소에 한 번이라도 올라갔다면, 이미 누군가 복사해 갔다고 가정해야 합니다.
2. **이력에서 제거합니다.** `git filter-repo` 같은 도구로 해당 내용을 이력에서 지우고, 원격 저장소에 강제로 반영합니다. 다른 팀원의 복제본, 포크, CI 캐시에도 남아 있을 수 있음을 알려야 합니다.
3. **재발 방지 장치를 둡니다.** 커밋 전에 자동으로 검사하는 pre-commit 훅이나 CI 단계를 둡니다(5단계, 8장 실습 1의 6단계 "파이프라인에 넣기").

> 🙋 (생각해 보기)
> 1번을 건너뛰고 2번만 하면 어떻게 될까요? 이력에서 지워도 **이미 복제된 저장소, 포크, CI 캐시, 검색 엔진에 남은 사본**은 지울 수 없습니다. 이력 제거는 "더 퍼지지 않게" 하는 일이고, 키를 무력화하는 것은 폐기뿐입니다.

## 5. 재발 방지: 커밋 전에 막기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | ✅ 완료 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | ✅ 완료 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | ✅ 완료 |
| 5 | 재발 방지: 커밋 전에 막기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | (선택) TruffleHog으로 교차 확인 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 5.1. Git 훅으로 커밋 전에 검사하기

Git은 커밋 직전에 `.git/hooks/pre-commit` 스크립트를 실행하고, 이 스크립트가 실패하면 커밋을 취소합니다. 여기에 Gitleaks를 걸어, **스테이징한 변경에 키가 있으면 커밋을 막습니다**.

> 📌 (참고) 팀 전체에 같은 훅을 배포할 때는 이 폴더의 `.pre-commit-config.yaml`처럼 [pre-commit](https://pre-commit.com) 도구를 씁니다. 여기서는 동작 원리를 보려고 훅 파일을 직접 만듭니다.

In [ ]:
print((LAB / ".pre-commit-config.yaml").read_text())

In [ ]:
%%bash
cd ~/leak-lab
cat > .git/hooks/pre-commit << 'EOF'
#!/bin/sh
# 스테이징한 변경만 검사하고, 키가 있으면 커밋을 막는다
gitleaks git --pre-commit --staged --redact -v --no-banner --no-color . || {
  echo "⛔ 비밀정보가 발견되어 커밋을 막았습니다. 키를 지우고 환경변수나 비밀 저장소를 쓰세요."
  exit 1
}
EOF
chmod +x .git/hooks/pre-commit
echo "훅 설치 완료: $(ls -l .git/hooks/pre-commit | awk '{print $1, $NF}')"

### 5.2. 키가 든 파일을 다시 커밋해 보기

In [ ]:
%%bash
cd ~/leak-lab
cat > rag_client.py << 'EOF'
EMBEDDING_API = "https://api.example-ai.com/v1/embeddings"
embedding_api_key = "sk-emb-Zx81Qa27Lp46Ws93Ed05Rf72Tg18Yh"
EOF
git add rag_client.py
git commit -m "RAG 임베딩 연동" 2>&1 || true
echo "── 커밋 이력 (새 커밋이 없어야 함)"
git log --oneline

훅이 키를 찾아 커밋을 막았습니다. 키를 환경변수로 바꾸면 통과합니다.

In [ ]:
%%bash
cd ~/leak-lab
cat > rag_client.py << 'EOF'
import os
EMBEDDING_API = "https://api.example-ai.com/v1/embeddings"
embedding_api_key = os.environ["EMBEDDING_API_KEY"]
EOF
git add rag_client.py
git commit -m "RAG 임베딩 연동" 2>&1 | tail -2
git log --oneline

> 🙋 (생각해 보기)
> 이 훅은 개발자 PC에만 있고, `git commit --no-verify`로 건너뛸 수도 있습니다. 그래서 훅은 "실수를 일찍 알려 주는" 장치이고, 반드시 막아야 하는 검사는 **CI(서버 쪽)에서 한 번 더** 합니다. 2장에서 본 "막을 것과 알릴 것"을 어디서 할지 정하는 문제입니다.

## 6. (선택) TruffleHog으로 교차 확인

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | ✅ 완료 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | ✅ 완료 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | ✅ 완료 |
| 5 | 재발 방지: 커밋 전에 막기 | ✅ 완료 |
| 6 | (선택) TruffleHog으로 교차 확인 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

TruffleHog은 발견한 키가 **실제로 유효한지** 해당 서비스에 확인하는 기능(`--only-verified`)이 있습니다. 설치 없이 Docker 이미지로 실행합니다. 처음에는 이미지를 받느라 조금 걸립니다.

In [ ]:
%%bash
docker pull -q trufflesecurity/trufflehog:3.88.0 >/dev/null
docker run --rm -v ~/leak-lab:/repo trufflesecurity/trufflehog:3.88.0 git file:///repo --no-update 2>&1 \
  | grep -vE '^\s*$' | tail -15

가짜 키는 `sk-proj-`처럼 실제 서비스의 형식을 흉내 냈지만, TruffleHog이 아는 탐지기(detector)와 맞지 않거나 검증에 실패하므로 결과가 없거나 "검증되지 않음(unverified)"으로 나옵니다. 실무에서는 **유효한(verified) 키부터 먼저 폐기**합니다.

> 📌 (참고) 두 도구는 찾는 방식이 다릅니다. Gitleaks는 정규식과 엔트로피로 "키처럼 생긴 값"을 넓게 찾고, TruffleHog은 서비스별 탐지기로 찾은 뒤 실제로 로그인해 보며 확인합니다. 하나만 믿지 말고 교차 확인하는 이유입니다.

## 7. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | ✅ 완료 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | ✅ 완료 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | ✅ 완료 |
| 5 | 재발 방지: 커밋 전에 막기 | ✅ 완료 |
| 6 | (선택) TruffleHog으로 교차 확인 | ✅ 완료 |
| 7 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 정리 | 예정 |

### 7.1. 체크포인트

- [ ] 최신 코드에서 지운 키가 저장소 이력 스캔에서 탐지되었나요?
- [ ] 이미 커밋된 키를 처리하는 올바른 순서(폐기 → 이력 제거 → 재발 방지)를 이해했나요?
- [ ] pre-commit 훅이 키가 든 커밋을 막는 것을 확인했나요?
- [ ] 우리 조직의 저장소 중 한 번도 이력 전체를 스캔해 본 적 없는 곳은 몇 개일까요?

### 7.2. 과제 1: 어떤 키가 어떤 규칙에 걸리나

AI 서비스에서 자주 쓰는 키 형식을 흉내 낸 **가짜 값**을 `keys.env`에 적고, Gitleaks가 어떤 규칙으로 잡는지 확인합니다. 아래 셀의 값을 바꾸거나 줄을 더해 보세요.

- 규칙 이름이 서비스 이름(`aws-access-token` 등)으로 나오는 것과 `generic-api-key`로 나오는 것은 무엇이 다를까요?
- 탐지되지 않은 줄이 있다면 이유는 무엇일까요? (3.3의 엔트로피 참고)

In [ ]:
fake_keys = {
    # 이름: 가짜 값 (실제 키 아님)
    # 이 노트북 파일 자체가 저장소 스캐너에 걸리지 않도록 앞부분을 나눠 적고 실행할 때 이어 붙입니다
    "AWS_ACCESS_KEY_ID": "AKIA" + "QX7RZK4MLP2TNW8E",
    "GITHUB_TOKEN": "ghp_" + "Xk29dLq81ZmW4tRb76NvYc03PsHe5Ja1GfUo",
    "SLACK_BOT_TOKEN": "xoxb-" + "2847562910-5837261049-Kd8Lm2Qx7Rt1Vb6Nz3Hy5Wc0",
    "DB_PASSWORD": "P@ssw0rd!",
}

tmp = Path("/tmp/keytype-test")
tmp.mkdir(exist_ok=True)
(tmp / "keys.env").write_text("".join(f"{k}={v}\n" for k, v in fake_keys.items()))
hits = {f["StartLine"]: f["RuleID"] for f in gitleaks_report("dir", tmp)}
table(["줄", "이름", "Gitleaks 규칙"],
      [(i, name, hits.get(i, "탐지 안 됨")) for i, name in enumerate(fake_keys, 1)])

### 7.3. 과제 2: 테스트용 가짜 키를 예외로 등록하기

테스트 코드에 일부러 넣은 가짜 키 때문에 CI가 매번 실패한다면 어떻게 해야 할까요? 규칙을 끄면 진짜 키도 놓칩니다. Gitleaks는 **특정 발견 항목 하나만** 예외로 두는 `.gitleaksignore` 파일을 지원합니다. 각 줄은 발견 항목의 지문(`Fingerprint`, `커밋:파일:규칙:줄`)입니다.

1. 아래 셀을 실행해 3단계에서 찾은 항목의 지문을 `.gitleaksignore`에 넣고 다시 검사합니다.
2. 이 예외 파일을 누가 검토해야 할지, 예외가 늘어나는 것을 어떻게 감시할지 적어 봅니다.

> 📌 (참고) 이 저장소의 루트에도 책 원고에 실은 예시 키를 예외로 둔 `.gitleaksignore`가 있습니다.

In [ ]:
before = gitleaks_report("git", REPO)
(REPO / ".gitleaksignore").write_text("".join(f["Fingerprint"] + "\n" for f in before))
print(".gitleaksignore 내용:\n" + (REPO / ".gitleaksignore").read_text())
after = gitleaks_report("git", REPO)
print(f"예외 등록 전 {len(before)}건 → 등록 후 {len(after)}건")

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

- 예외 파일을 검토할 사람:
- 예외가 늘어나는 것을 감시하는 방법:

### 7.4. 과제 3: 사고 대응 메시지 쓰기 (서술)

사내 Git 서버의 공개 저장소에서 고객지원 챗봇의 API Key가 3개월 전 커밋에 들어 있는 것을 발견했습니다. 4단계의 순서에 따라 **개발팀에 보낼 조치 요청을 다섯 줄 이내로** 써 보세요. 누가, 무엇을, 어떤 순서로 해야 하는지 들어가야 합니다.

✍️ **답 적기**

1.
2.
3.

## 8. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: Gitleaks 확인 | ✅ 완료 |
| 1 | 실습용 저장소 만들고 가짜 키 커밋하기 | ✅ 완료 |
| 2 | 실수를 "고친" 척하기 | ✅ 완료 |
| 3 | 현재 파일 스캔과 이력 전체 스캔 비교 | ✅ 완료 |
| 4 | 이미 커밋된 키는 어떻게 처리하나 | ✅ 완료 |
| 5 | 재발 방지: 커밋 전에 막기 | ✅ 완료 |
| 6 | (선택) TruffleHog으로 교차 확인 | ✅ 완료 |
| 7 | 체크포인트와 과제 | ✅ 완료 |
| 8 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [ ]:
%%bash
rm -rf ~/leak-lab /tmp/entropy-test /tmp/keytype-test /tmp/gitleaks-report.json
docker rmi trufflesecurity/trufflehog:3.88.0 >/dev/null 2>&1
echo "정리 완료"

---
> 4장의 손 실습(실습 1 데이터 처리 옵션, 실습 2 사용 유형 자가진단)은 책의 [실습 상세 가이드]와 이 폴더의 [README.md](./README.md)에 있습니다. 키를 코드 대신 어디에 두고 어떻게 중계하는지는 6장 실습 2(가명처리 검문소)와 7장 실습 2에서 다룹니다.